# Reviews: Data Cleaning

Each row represents one review. `id` uniquely identifies the review, and `listing_id` links it to a listing.

| Feature | Description | Decision |
|---|---|---|
| `listing_id` | Listing ID; links to `listings.id` | Keep as a string |
| `id` | Unique review ID | Keep as a string |
| `date` | Review date (YYYY-MM-DD) | Keep; convert to a date |
| `reviewer_id` | Reviewer ID; identifies repeat reviewers | Keep as a string |
| `reviewer_name` | Reviewer's display name | Remove; the reviewer ID is sufficient for analysis |
| `comments` | Review text in its original language | Keep; replace HTML line breaks and trim surrounding whitespace |

**Removed feature: `reviewer_name`.** No review text is translated, and no ratings or sentiment labels are inferred. Reviews with the same listing, reviewer and date are retained when their review IDs differ.


## Load the File


In [1]:
from pathlib import Path
import pandas as pd

from utils.constants import DATA_PATH

AIRBNB_DATA = DATA_PATH / "airbnb"
PROCESSED_DATA = AIRBNB_DATA / "processed"

FILE_PATH = AIRBNB_DATA / "reviews.csv"

df = pd.read_csv(FILE_PATH, dtype="string", keep_default_na=False)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())

Rows: 166,490 | Columns: 6


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,145320,156423122,2017-05-30,123386382,Erwin,Prima plek om Stockholm te bekijken. Alles is ...
1,145320,170211906,2017-07-15,123091743,Anne,Cosy and clean flat in quiet neighbourhood clo...
2,145320,176647581,2017-07-31,103178743,Peter,Kim's place was outstanding and comfortable. W...
3,145320,185676021,2017-08-22,4023961,Alexander,great spacious apartment in a nice residential...
4,145320,189668224,2017-09-02,142869362,Heiko,Kim is a very friendly person.A lot of advice ...


## 1. Inspect Missing Values and Duplicates

Empty or whitespace-only fields count as missing. Duplicate review IDs would require inspection before export. Matching listing, reviewer and date combinations alone are not sufficient evidence for deleting a review.

In [2]:
profile = pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Missing or blank values": df.apply(lambda column: (column.isna() | column.str.strip().eq("")).sum()),
    "Unique values": df.nunique(),
})
display(profile)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate review IDs:", int(df.duplicated("id").sum()))
print("Additional rows with the same listing, reviewer and date:",
      int(df.duplicated(["listing_id", "reviewer_id", "date"]).sum()))

,Data type,Missing or blank values,Unique values
listing_id,string,0,3912
id,string,0,166490
date,string,0,4721
reviewer_id,string,0,157057
reviewer_name,string,0,31975
comments,string,0,162355


Exact duplicate rows: 0
Duplicate review IDs: 0
Additional rows with the same listing, reviewer and date: 8


## 2. Remove the Reviewer Name

The display name is not needed for the planned analysis. `reviewer_id` remains available to distinguish reviewers.

In [3]:
df = df.drop(columns=["reviewer_name"])
display(df.head())

,listing_id,id,date,reviewer_id,comments
0,145320,156423122,2017-05-30,123386382,Prima plek om Stockholm te bekijken. Alles is ...
1,145320,170211906,2017-07-15,123091743,Cosy and clean flat in quiet neighbourhood clo...
2,145320,176647581,2017-07-31,103178743,Kim's place was outstanding and comfortable. W...
3,145320,185676021,2017-08-22,4023961,great spacious apartment in a nice residential...
4,145320,189668224,2017-09-02,142869362,Kim is a very friendly person.A lot of advice ...


## 3. Convert the Review Date

Invalid dates stop execution instead of silently becoming missing values.

In [4]:
assert df["date"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all(), "Unexpected date format."

df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise")
print("Review period:", df["date"].min().date(), "to", df["date"].max().date())

Review period: 2011-07-30 to 2026-07-03


## 4. Clean Text Formatting

Replace HTML line break tags such as `<br/>` with actual line breaks and remove surrounding whitespace. Preserve the language, punctuation and wording of each review.

In [5]:
original_comments = df["comments"].copy()
print("Reviews containing HTML line breaks:",
      int(original_comments.str.contains(r"<br\s*/?>", case=False, regex=True).sum()))
print("Reviews with surrounding whitespace:",
      int(original_comments.ne(original_comments.str.strip()).sum()))

df["comments"] = (df["comments"]
    .str.replace(r"<br\s*/?>", "\n", case=False, regex=True)
    .str.strip())
changed = original_comments.ne(df["comments"])
print("Reviews with updated text formatting:", int(changed.sum()))
display(pd.DataFrame({
    "Before": original_comments[changed],
    "After": df.loc[changed, "comments"],
}).head(3))

Reviews containing HTML line breaks: 36185
Reviews with surrounding whitespace: 3217
Reviews with updated text formatting: 38802


,Before,After
0,Prima plek om Stockholm te bekijken. Alles is ...,Prima plek om Stockholm te bekijken. Alles is ...
11,Very pleasant stay. We recommend it!!<br/>Kim ...,Very pleasant stay. We recommend it!!\nKim is ...
16,I would definitely recommend Kim's place for a...,I would definitely recommend Kim's place for a...


## 5. Check the Result and Export

All review rows are retained. CSV files do not store pandas data types: read IDs as strings and parse `date` when loading the exported file again. The output is saved in the notebook's working directory.

In [6]:
assert not df.isna().any().any(), "Unexpected missing values."
assert not df["comments"].str.strip().eq("").any(), "Inspect empty review text."
assert not df["id"].duplicated().any(), "Unexpected duplicate review IDs."
display(df.dtypes.astype(str).to_frame("Final data type"))
print(f"Final rows: {len(df):,} | Final columns: {df.shape[1]}")

df.to_csv(PROCESSED_DATA / "processed_reviews_data.csv", index=False)

,Final data type
listing_id,string
id,string
date,datetime64[us]
reviewer_id,string
comments,string


Final rows: 166,490 | Final columns: 5
